In [5]:
 %load_ext autoreload
%autoreload 2
import numpy as np
from utils_final_results import load_dataset
from hydra import compose
from sklearn.linear_model import Lasso, LassoCV
from sklearn.metrics import mean_squared_error
from utils_table_generator import get_mean_std, flatten_config
from omegaconf import OmegaConf

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
def construct_dataset(dataset):
    X_list = []
    y_list = []
    sex_list = []
    mutation_list = []
    age_list = []

    for data in dataset:
        graph_feat = data.x.view(-1).numpy()
        X_list.append(graph_feat)
        y_list.append(data.y.item())
        sex_list.append(data.sex.item())
        mutation_list.append(data.mutation.item())
        age_list.append(data.age.item())

    X = np.vstack(X_list)
    y = np.array(y_list)

    sex = np.array(sex_list).reshape(-1, 1)
    mutation = np.array(mutation_list).reshape(-1, 1)
    age = np.array(age_list).reshape(-1, 1)

    return X, y, sex, mutation, age


def stack_features(train_features, train_labels, train_sex, train_mutation, train_age, val_features, val_labels, val_sex, val_mutation, val_age):
    target_shape = train_features.shape[1] // 3
    if train_sex.shape[1] < target_shape:
        # Calculate how many times to repeat train_sex along the second axis
        repeat_times = target_shape // train_sex.shape[1]
        train_sex = np.tile(train_sex, (1, repeat_times))
        val_sex = np.tile(val_sex, (1, repeat_times))
        train_mutation = np.tile(train_mutation, (1, repeat_times))
        val_mutation = np.tile(val_mutation, (1, repeat_times))
        train_age = np.tile(train_age, (1, repeat_times))
        val_age = np.tile(val_age, (1, repeat_times))

    train_combined = np.hstack((train_features, train_sex, train_mutation, train_age))
    print(train_combined.shape)
    val_combined = np.hstack((val_features, val_sex, val_mutation, val_age))
    print(val_combined.shape)
    return train_combined, val_combined, train_labels, val_labels

def concat_features(train_features, train_labels, train_sex, train_mutation, train_age, val_features, val_labels, val_sex, val_mutation, val_age):
    train_combined = np.hstack((train_features, train_sex, train_mutation, train_age))
    val_combined = np.hstack((val_features, val_sex, val_mutation, val_age))
    return train_combined, val_combined, train_labels, val_labels

def convert_results(norm_mean, norm_std, adj_metric, adj_thresh, k_fold, num_folds, fold) :
    cfg = compose(
        config_name="run.yaml",
        overrides=[
            "model=graph/gcn",
            "dataset=graph/FTD",
            f"dataset.loader.parameters.adj_metric={adj_metric}",
            f"dataset.loader.parameters.adj_thresh={adj_thresh}",
            f"dataset.loader.parameters.kfold={k_fold}",
            f"dataset.loader.parameters.num_folds={num_folds}",
            f"dataset.loader.parameters.fold={fold}",
        ],
        return_hydra_config=True
    )
    cfg_dict = OmegaConf.to_container(cfg, resolve=False)
    cfg_flat = flatten_config(cfg_dict)
    mean, std = get_mean_std(cfg_flat)
    assert mean is not None, "Mean is None for run"
    assert std is not None, "Std is None for run"
    orig_mean = norm_mean * std**2
    orig_std = norm_std * std**2
    return orig_mean, orig_std


def run_lasso_folds(num_folds=5, k_fold=True, expand_features=True, adj_metric="wgcna", adj_thresh=0.5):
    alphas = np.logspace(-4, 2, 50) # Range of alpha values to try
    # Dictionary to store MSE scores for each alpha
    alpha_results = {alpha: [] for alpha in alphas}
    
    for fold in range(num_folds):
        [train, val, test] = load_dataset(adj_metric, adj_thresh, k_fold, num_folds, fold=fold)
        train_features, train_labels, train_sex, train_mutation, train_age = construct_dataset(train)
        val_features, val_labels, val_sex, val_mutation, val_age = construct_dataset(val)
        if expand_features:
            train_combined, val_combined, train_labels, val_labels = stack_features(train_features, train_labels, train_sex, train_mutation, train_age, val_features, val_labels, val_sex, val_mutation, val_age)
        else:
            train_combined, val_combined, train_labels, val_labels = concat_features(train_features, train_labels, train_sex, train_mutation, train_age, val_features, val_labels, val_sex, val_mutation, val_age)
    
        # Try different alpha values
        for alpha in alphas:
            lasso_model = Lasso(alpha=alpha, random_state=42)
            lasso_model.fit(train_combined, train_labels)
            
            # Predict and evaluate
            predictions = lasso_model.predict(val_combined)
            mse = mean_squared_error(val_labels, predictions)
            alpha_results[alpha].append(mse)
        
        print(f"Completed fold {fold}")
    
    # Calculate and print statistics for each alpha
    print("\nResults for each alpha value across folds:")
    print("Alpha\tMean MSE ± Std MSE\tMean MSE (orig units) ± Std MSE (orig units)")
    print("-" * 30)
    
    best_mean_mse = float('inf')
    best_alpha = None
    
    for alpha in alphas:
        mean_mse = np.mean(alpha_results[alpha])
        std_mse = np.std(alpha_results[alpha])
        orig_mean, orig_std = convert_results(mean_mse, std_mse, adj_metric, adj_thresh, k_fold, num_folds, fold)
        print(f"{alpha:.3f}\t{mean_mse:.4f} ± {std_mse:.4f}\t{orig_mean:.4f} ± {orig_std:.4f}")
        
        if mean_mse < best_mean_mse:
            best_mean_mse = mean_mse
            best_alpha = alpha
    
    print(f"\nBest alpha across all folds: {best_alpha} (MSE: {best_mean_mse:.4f})")
    
    return alpha_results

In [7]:
run_lasso_folds(num_folds=5, expand_features=True)

Processed file names: ['FTD_y_val_nfl_wgcna_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_train.pt', 'FTD_y_val_nfl_wgcna_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_val.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_nfl_wgcna_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_train.pt
Processed file names: ['FTD_y_val_nfl_wgcna_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_train.pt', 'FTD_y_val_nfl_wgcna_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_val.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_nfl_wgcna_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_val.pt
Processed file names: ['FTD_y_val_nfl_wgcna_adj_thresh_0.5_n

{np.float64(0.0001): [np.float64(0.733335499608077),
  np.float64(1.1078785909760072),
  np.float64(1.746015263673172),
  np.float64(0.5711348155178838),
  np.float64(0.5517629494400959)],
 np.float64(0.00013257113655901095): [np.float64(0.7296098372296913),
  np.float64(0.8716640645179942),
  np.float64(1.7363335071444768),
  np.float64(0.640507862255903),
  np.float64(0.4909915214916394)],
 np.float64(0.00017575106248547912): [np.float64(0.6939976499976762),
  np.float64(0.7758839634594251),
  np.float64(1.6998477487870303),
  np.float64(0.6794901207517058),
  np.float64(0.4658066586835991)],
 np.float64(0.00023299518105153718): [np.float64(0.6410896734209071),
  np.float64(0.7106191802596916),
  np.float64(1.6940250932216327),
  np.float64(0.6957101964710755),
  np.float64(0.5328620864804858)],
 np.float64(0.00030888435964774815): [np.float64(0.6505950564236169),
  np.float64(0.6695439703088423),
  np.float64(1.7318498908721756),
  np.float64(0.715174358444991),
  np.float64(0.71208